In [0]:
%pip install -U --quiet \
    databricks-langchain \
    langchain \
    langchain-community \
    wikipedia \
    youtube_search \
    duckduckgo-search\
    -U ddgs \
    -U langgraph

dbutils.library.restartPython()

In [0]:
import mlflow

# Set experiment for better organization
mlflow.set_experiment("/Users/naval.datamaster@gmail.com/agent")

# Enable autologging BEFORE creating the LangChain client
mlflow.langchain.autolog()


# Why Use Frameworks like LangChain & LangGraph?

Even though you *can* write plain Python, frameworks help with things that get complicated fast:

- **Workflow Management:** Organize multi-step tasks, agents, and APIs cleanly.  
- **State Management:** Keep track of variables, context, or intermediate results across steps.  
- **Memory:** Remember past interactions or user inputs for smarter responses.  
- **Retries & Error Handling:** Automatically retry failed steps or handle exceptions.  
- **Reusability & Maintainability:** Reuse components, swap models/tools without rewriting everything.  
- **Dynamic Logic & Branching:** Easily implement loops, conditional paths, and agent collaboration.  


In [0]:
from databricks_langchain import ChatDatabricks

# Initialize model
llm = ChatDatabricks(endpoint="databricks-gpt-oss-120b")

print(llm.invoke("What is Databricks?"))

# 🧩 Simple StateGraph Workflow — Concepts Explained

This code creates a **state-based workflow** using LangGraph. Instead of calling functions directly in sequence, we define:

- A **State**: a typed dictionary (`TypedDict`) that describes what data flows between nodes (`graph_state` in this case).
- **Nodes**: plain Python functions (`node_1`, `node_2`, `node_3`) that:
  - Receive the current `state` (a dict)
  - Perform some operation (here, just string concatenation)
  - Return an updated `state` (partial dict with changed values)

- A **StateGraph**: an object that stores the nodes and how they connect.
  - We add each node to the graph by name.
  - We define **edges** between nodes — this describes the execution path (`START → node_1 → node_2 → node_3 → END`).

- **Compilation**: `builder.compile()` turns the defined graph into an executable workflow.
- **Visualization**: `graph.get_graph().draw_mermaid_png()` renders a diagram of the graph so you can see the structure visually in the notebook.

**Conceptually:**  
This pattern abstracts workflows into **directed graphs**. Each node is independent, the graph controls the order, and the shared `state` moves through the graph — a foundation for building more complex, branching, or parallel agent systems.


In [0]:
#from dotenv import load_dotenv
import os
import random
from typing import Literal, TypedDict
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph.message import add_messages
from typing_extensions import Annotated
from databricks_langchain import ChatDatabricks
from langgraph.graph import StateGraph, START, END
#from langgraph.prebuilt import ToolNode, tools_condition
from IPython.display import Image, display



from typing_extensions import TypedDict

class State(TypedDict):
    graph_state: str

def node_1(state):
    print("---Node 1---")
    return {"graph_state": state['graph_state'] +" I am"}

def node_2(state):
    print("---Node 2---")
    return {"graph_state": state['graph_state'] +" happy!"}

def node_3(state):
    print("---Node 3---")
    return {"graph_state": state['graph_state'] +" sad!"}



# Build graph
builder = StateGraph(State)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)





# Logic
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)

# Add
graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png(max_retries=5, retry_delay=2.0)))



In [0]:
graph.invoke({"graph_state" : "Hi, this is Naval."})


In [0]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict


# define state
class BMIState(TypedDict):

    weight_kg: float
    height_m: float
    bmi: float
    category: str

#defined node 1
def calculate_bmi(state: BMIState) -> BMIState:

    weight = state['weight_kg']
    height = state['height_m']

    bmi = weight/(height**2)

    state['bmi'] = round(bmi, 2)

    return state

#defined node 2
def label_bmi(state: BMIState) -> BMIState:

    bmi = state['bmi']

    if bmi < 18.5:
        state["category"] = "Underweight"
    elif 18.5 <= bmi < 25:
        state["category"] = "Normal"
    elif 25 <= bmi < 30:
        state["category"] = "Overweight"
    else:
        state["category"] = "Obese"

    return state

In [0]:
# define your graph
graph = StateGraph(BMIState)

# add nodes to your graph
graph.add_node('calculate_bmi', calculate_bmi)
graph.add_node('label_bmi', label_bmi)

# add edges to your graph
graph.add_edge(START, 'calculate_bmi')
graph.add_edge('calculate_bmi', 'label_bmi')
graph.add_edge('label_bmi', END)


# compile the graph
workflow = graph.compile()

In [0]:
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())

In [0]:
# execute the graph
intial_state = {'weight_kg':70, 'height_m':1.73}

final_state = workflow.invoke(intial_state)

print(final_state)

In [0]:
from langgraph.graph import StateGraph, START, END
from databricks_langchain import ChatDatabricks
from typing import TypedDict

llm = ChatDatabricks(endpoint="databricks-gpt-oss-120b")


# create a state
class LLMState(TypedDict):
    question: str
    answer: str


def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = llm.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state


# create our graph

graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()


# execute

intial_state = {'question': 'How far is moon from the earth?'}

final_state = workflow.invoke(intial_state)

print(final_state['answer'])